# Module 6 / Class 1 -- Neural Network Fundamentals

**Objectives:**
- Build a single perceptron from scratch using NumPy
- Understand: inputs, weights, bias, activation function, output
- See matrix multiplication in action (forward pass)
- Connect concepts to TensorFlow Playground

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. Single Perceptron from Scratch

A perceptron computes:

$$y = \sigma(w_1 x_1 + w_2 x_2 + b)$$

where $\sigma$ is an activation function.

In [ ]:
def sigmoid(z):
    """Sigmoid activation function."""
    return 1.0 / (1.0 + np.exp(-z))


def perceptron(x, w, b):
    """Single perceptron: weighted sum + bias + activation."""
    z = np.dot(x, w) + b          # linear combination
    return sigmoid(z)              # activation


# Example: AND gate
# Inputs: two binary features
inputs = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
])

# Hand-picked weights that approximate AND
w = np.array([10.0, 10.0])
b = -15.0

print("AND gate with a perceptron:")
print(f"{'x1':>3} {'x2':>3} | {'z':>8} | {'output':>8} | {'rounded':>8}")
print("-" * 45)
for x in inputs:
    z = np.dot(x, w) + b
    out = sigmoid(z)
    print(f"{x[0]:>3} {x[1]:>3} | {z:>8.2f} | {out:>8.4f} | {round(out):>8}")

In [ ]:
# Visualize the sigmoid function
z_vals = np.linspace(-10, 10, 200)
sig_vals = sigmoid(z_vals)

plt.figure(figsize=(7, 4))
plt.plot(z_vals, sig_vals, linewidth=2, color='steelblue')
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.title('Sigmoid Activation Function')
plt.xlabel('z')
plt.ylabel('sigmoid(z)')
plt.grid(True, alpha=0.3)
plt.show()

## 2. OR Gate -- Adjust Weights Yourself

The AND gate uses w=[10, 10], b=-15. The OR gate needs different weights.

In [ ]:
# OR gate weights
w_or = np.array([10.0, 10.0])
b_or = -5.0

print("OR gate with a perceptron:")
print(f"{'x1':>3} {'x2':>3} | {'output':>8} | {'rounded':>8}")
print("-" * 35)
for x in inputs:
    out = perceptron(x, w_or, b_or)
    print(f"{x[0]:>3} {x[1]:>3} | {out:>8.4f} | {round(out):>8}")

## 3. The XOR Problem -- Why One Perceptron Is Not Enough

XOR cannot be solved by a single perceptron because the classes are not linearly separable.

In [ ]:
# XOR truth table
xor_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
xor_targets = np.array([0, 1, 1, 0])

# Visualize -- no single line can separate the classes
plt.figure(figsize=(5, 5))
for i in range(4):
    color = 'red' if xor_targets[i] == 1 else 'blue'
    plt.scatter(xor_inputs[i, 0], xor_inputs[i, 1], c=color, s=200,
                edgecolors='black', linewidths=2, zorder=5)
plt.title('XOR Problem -- Not Linearly Separable')
plt.xlabel('x1')
plt.ylabel('x2')
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 1.5)
plt.ylim(-0.5, 1.5)
plt.show()

print("No single straight line can separate red from blue. We need a hidden layer.")

## 4. Two-Layer Network for XOR (from scratch)

Architecture: 2 inputs -> 2 hidden neurons -> 1 output neuron

In [ ]:
# Hand-crafted weights that solve XOR
# Hidden layer: 2 neurons
W1 = np.array([[20.0, -20.0],
               [20.0, -20.0]])
b1 = np.array([-10.0, 30.0])

# Output layer: 1 neuron
W2 = np.array([[20.0],
               [20.0]])
b2 = np.array([-30.0])


def forward_pass(x):
    """Forward pass through the 2-layer network."""
    # Hidden layer
    z1 = x @ W1 + b1
    h = sigmoid(z1)
    # Output layer
    z2 = h @ W2 + b2
    y = sigmoid(z2)
    return y, h


print("XOR with a 2-layer network:")
print(f"{'x1':>3} {'x2':>3} | {'h1':>6} {'h2':>6} | {'output':>8} | {'target':>7}")
print("-" * 50)
for i, x in enumerate(xor_inputs):
    y, h = forward_pass(x)
    print(f"{x[0]:>3} {x[1]:>3} | {h[0]:>6.3f} {h[1]:>6.3f} | {y[0]:>8.4f} | {xor_targets[i]:>7}")

## 5. Matrix Multiplication Demo -- Batch Forward Pass

In practice, we process all inputs at once using matrix multiplication. This is much faster than looping.

In [ ]:
# All 4 XOR inputs at once
print("Input matrix X (4 samples, 2 features):")
print(xor_inputs)
print(f"Shape: {xor_inputs.shape}")
print()

# Hidden layer: X @ W1 + b1
Z1 = xor_inputs @ W1 + b1
print("Z1 = X @ W1 + b1:")
print(Z1)
print(f"Shape: {Z1.shape}")
print()

H = sigmoid(Z1)
print("H = sigmoid(Z1):")
print(np.round(H, 4))
print()

# Output layer: H @ W2 + b2
Z2 = H @ W2 + b2
Y = sigmoid(Z2)
print("Output Y = sigmoid(H @ W2 + b2):")
print(np.round(Y, 4))
print()
print("Rounded outputs:", np.round(Y.flatten()).astype(int))
print("Targets:        ", xor_targets)

## 6. Common Activation Functions

In [ ]:
def relu(z):
    return np.maximum(0, z)

def tanh(z):
    return np.tanh(z)

z = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(z, sigmoid(z), linewidth=2, color='steelblue')
axes[0].set_title('Sigmoid')
axes[0].set_ylim(-0.1, 1.1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(z, tanh(z), linewidth=2, color='coral')
axes[1].set_title('Tanh')
axes[1].grid(True, alpha=0.3)

axes[2].plot(z, relu(z), linewidth=2, color='seagreen')
axes[2].set_title('ReLU')
axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('z')
    ax.set_ylabel('f(z)')
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

## 7. TensorFlow Playground

Go to [TensorFlow Playground](https://playground.tensorflow.org/) and experiment with:

1. **XOR dataset** (bottom-right pattern): Try 1 hidden layer with 2 neurons. Does it converge? Try adding more neurons.
2. **Spiral dataset**: How many hidden layers and neurons do you need?
3. **Activation functions**: Compare ReLU vs Sigmoid vs Tanh. Which converges faster?
4. **Learning rate**: What happens when it is too high? Too low?

---

## TODO: Student Work

### Document your TensorFlow Playground findings

For each experiment, note:
- What configuration you used (layers, neurons, activation, learning rate)
- What happened (did it converge? how many epochs?)
- What you learned

# TODO: Write your findings here

**Experiment 1 -- XOR dataset:**
- Configuration: ...
- Result: ...
- Insight: ...

**Experiment 2 -- Spiral dataset:**
- Configuration: ...
- Result: ...
- Insight: ...

**Experiment 3 -- Activation functions:**
- Configuration: ...
- Result: ...
- Insight: ...

**Experiment 4 -- Learning rate:**
- Configuration: ...
- Result: ...
- Insight: ...